# __Lecture 17:__ working with immune repertoire sequencing data

Today we'll work with IMGT/V-QUEST AIRR-formatted annotations of human antibody heavy chain sequences.

This is the fourth sequencing data type you've seen in the last few weeks, and it has the same shape as the others: a set of sequences, a table of annotations describing them, and a coordinate system tying the two together. FASTQ/SAM in lecture 15, genomic ranges in lecture 16, and now AIRR. 

Our goals:

1. Load and inspect AIRR-formatted sequence annotations
2. Compute per-sequence mutation counts by comparing observed vs germline sequences
3. Compare mutation density across the regions of the V gene using IMGT boundaries
4. (if time) Look for mutation hotspot motifs using regular expressions

In [ ]:
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 0. Loading and inspecting the data

I've annotated 50 human antibody heavy chain sequences using [IMGT/V-QUEST](https://www.imgt.org/IMGT_vquest/analysis).

Today we will explore these annotated sequences and eventually quantify how many mutations occur in different antibody regions.

Let's begin by loading the dataset and looking at a few key annotation fields.

We will focus on:

- `sequence_alignment`: observed nucleotide sequence aligned to a reference
- `germline_alignment`: inferred germline sequence aligned to the same reference
- `cdr*_start` / `cdr*_end`: boundaries of the CDR regions. These are **1-based indices** referring to the *ungapped* germline numbering scheme.
- `fwr*_start` / `fwr*_end`: boundaries of the framework regions

One scoping note. Today we will only look at mutations in the V gene, which runs from the start of FWR1 to the end of FWR3. Past that point the sequence crosses the junction, where recombination deleted nucleotides from the gene ends and inserted random ones between them. Those inserted nucleotides have no germline counterpart, so IMGT writes `n` for the germline there. They are differences from the germline, but they were produced by recombination rather than by mutation, so counting them would answer a different question than the one we're asking.

In [ ]:
# Load the AIRR-formatted annotations from IMGT/V-QUEST
df = pd.read_csv("vquest_airr.tsv", sep="\t")

# Look at the first few rows to see what the table looks like
df.head()

### Subsetting to the columns we will use

The IMGT output contains many fields (~125 columns).
For this tutorial, let's keep only the columns needed for mutation counting and region analysis.

In [ ]:
cols = [
    "sequence_id",
    "sequence_alignment",
    "germline_alignment",
    "cdr1_start",
    "cdr1_end",
    "cdr2_start",
    "cdr2_end",
    "fwr1_start",
    "fwr1_end",
    "fwr2_start",
    "fwr2_end",
    "fwr3_start",
    "fwr3_end",
]

df_subset = df[cols].copy()

df_subset.head()

### Inspecting the first sequence

Before writing any code to count mutations, let's look closely at the aligned observed and germline sequences for the first entry.

This will help us understand the data formatting and decide how to compare aligned sequences properly.

In [ ]:
first_seq = df_subset.iloc[0]

print("Germline alignment:\n", first_seq["germline_alignment"])
print("\nObserved alignment:\n", first_seq["sequence_alignment"])

__Question:__ Do you see positions where the nucleotides differ?

Notice that IMGT uses `"."` characters as **numbering spacers** so that all sequences share a unified coordinate system (IMGT's standardized V-region numbering). These `"."` symbols are **not alignment gaps** and do **not** indicate identity with the germline. Instead, **they are positional placeholders**.

For our purposes today, these `"."` characters would interfere with counting mutations and with using CDR/FWR boundary coordinates. To keep things simple, we will remove all `"."` characters from both the observed and germline alignments.

In [ ]:
# Remove IMGT numbering spacer dots so indices match CDR/FWR boundaries
df_subset["germline_alignment"] = df_subset["germline_alignment"].str.replace(
    ".", "", regex=False
)
df_subset["sequence_alignment"] = df_subset["sequence_alignment"].str.replace(
    ".", "", regex=False
)

df_subset.head()

### Trimming to the V gene

As described above, we're restricting today's analysis to the V gene, which ends at `fwr3_end`. Cutting both alignments there once, now, keeps everything downstream simple: every position we look at from here on has a real germline nucleotide to compare against.

The region coordinates still work after trimming, because FWR1 starts at position 1 and we're only removing sequence from the far end.

In [ ]:
# Keep only the V gene portion of each alignment
df_subset["germline_alignment"] = [
    germ[:end]
    for germ, end in zip(df_subset["germline_alignment"], df_subset["fwr3_end"])
]
df_subset["sequence_alignment"] = [
    seq[:end]
    for seq, end in zip(df_subset["sequence_alignment"], df_subset["fwr3_end"])
]

df_subset.head()

## 1. Counting total mutations per sequence

To begin, we will count the total number of nucleotide mutations in each sequence.

We will treat each position where the observed nucleotide differs from the germline nucleotide as a mutation.

Because we already removed the `"."` spacer characters from the alignments, the observed and germline sequences now:

- have the same length
- align position-by-position

In [ ]:
def count_mutations(seq_align, germ_align):
    """Count nucleotide mismatches between aligned query and germline sequences."""
    mismatches = 0

    for q, g in zip(seq_align, germ_align):
        if q != g:
            mismatches += 1

    return mismatches

Let's try this function on the first sequence to confirm it works as expected.

In [ ]:
first_seq = df_subset.iloc[0]

first_mutations = count_mutations(
    first_seq["sequence_alignment"], first_seq["germline_alignment"]
)

first_mutations

### Exercise 1 (approx. 10 minutes)

**Goal:** Compute the number of V gene mutations for each sequence and visualize the distribution.

Steps:

1. Loop over all rows in `df_subset` and use `count_mutations()` to compute the mutation count
   *(Hint: use `df_subset.iterrows()`)*
2. Save the result to a new column named `mutation_count`
3. Use `seaborn.histplot` to visualize the distribution, and compute the mean and median

A loop is the clearest way to write this, and it's what we'll use. If you'd rather write it the way you saw in lectures 10-13, `df_subset.apply(..., axis=1)` in pandas and `mutate()` in dplyr both do the same job in one line.

In [ ]:
# Steps 1–2: compute mutation counts

# your code here...

In [ ]:
# Step 3: visualize the distribution, and summarize it

# your code here...

## 2. Where in the antibody do those mutations fall?

A single number per sequence tells us how mutated an antibody is, but not *where* the mutations are.

Somatic hypermutation does not fall evenly along the sequence. It tends to target the **CDR regions**, which form the loops that contact antigen, more heavily than the **framework (FWR) regions**, which hold those loops in place. The V gene alternates between the two: FWR1, CDR1, FWR2, CDR2, FWR3.

To see that, we need to count mutations inside a stretch of the sequence rather than across all of it. That is the same job as before applied to a shorter pair of strings, so the only new thing to work out is how to cut the region out.

### Exercise 2 (approx. 10 minutes)

**Goal:** Write a function that counts mutations in one region.

IMGT coordinates are **1-based and inclusive**. Python slices are **0-based and exclude the end**. So the region running from `start` to `end` is `seq[start - 1 : end]`.

Getting a region out of a sequence by its start and end coordinates is the same operation you did in lecture 16 with genomic ranges, except there a package handled the indexing for you. Off-by-one errors here are the most common bug in this kind of analysis, so the cell after this one checks your work: asking for the region from 1 to the full sequence length should give back the same answer as `count_mutations()` on the whole sequence.

In [ ]:
def count_region_muts(seq_align, germ_align, start, end):
    """Count mutations between seq and germ within one region."""
    # your code here: cut both sequences down to the region,
    # then hand the pieces to count_mutations()

    return None

In [ ]:
# Check: over the full length of a sequence, the two functions should agree
check_seq = df_subset.iloc[0]
whole = len(check_seq["sequence_alignment"])

full_seq_mutations = count_mutations(
    check_seq["sequence_alignment"], check_seq["germline_alignment"]
)

full_seq_mutations2 = count_region_muts(
    check_seq["sequence_alignment"], check_seq["germline_alignment"], 1, whole
)

assert (
    full_seq_mutations == full_seq_mutations2
), "region count does not match full-length count"

### Applying it to every region

Now we use your function on all five regions of all 49 sequences.

Region lengths differ, so raw counts aren't comparable between them: a long region collects more mutations simply by being long. What we want is **mutation density**, the number of mutations divided by the length of the region.

The cell below does that, using your function. Read it, run it, then plot the result.

In [ ]:
# The regions of the V gene, in the order they appear along the sequence
regions = {
    "FWR1": ("fwr1_start", "fwr1_end"),
    "CDR1": ("cdr1_start", "cdr1_end"),
    "FWR2": ("fwr2_start", "fwr2_end"),
    "CDR2": ("cdr2_start", "cdr2_end"),
    "FWR3": ("fwr3_start", "fwr3_end"),
}

densities = pd.DataFrame()

for region_name, (start_col, end_col) in regions.items():
    counts = [
        count_region_muts(
            seq["sequence_alignment"],
            seq["germline_alignment"],
            seq[start_col],
            seq[end_col],
        )
        for _, seq in df_subset.iterrows()
    ]
    lengths = df_subset[end_col] - df_subset[start_col] + 1
    densities[region_name] = counts / lengths

densities.head()

In [ ]:
# One box per region. `densities` has one column per region, which is the shape
# seaborn expects, so this needs no reshaping.
sns.boxplot(data=densities)
plt.ylabel("mutation density")
plt.show()

### Questions

Based on your plot:

- Which regions have the highest mutation density? Do they alternate the way you'd expect?
- Does the pattern match the biology, given that CDRs contact antigen directly and frameworks provide structural scaffolding?
- Look at the spread within each region, not just the averages. How consistent is the pattern from sequence to sequence? With 49 sequences, how confident would you be in this result?

## 3. (Optional) Mutation hotspot motifs

Somatic hypermutation is not uniform along the sequence. It is initiated by AID, which prefers particular short sequence contexts. The classic hotspot motifs are written **WRC** and its reverse complement **GYW**, using IUPAC ambiguity codes: W is A or T, R is A or G, Y is C or T.

Those codes are character classes, so WRC is the regular expression `[AT][AG]C` and GYW is `G[CT][AT]`. This is the `re` module from lecture 08, used on a real question.

**Goal:** check whether mutated positions sit in hotspot motifs more often than other positions do.

Steps:

1. Walk every position of every germline sequence, skipping the first and last, and pull out the 3-mer centered on it
2. Sort the position into one of two groups: hotspot, if the 3-mer matches `[AT][AG]C|G[CT][AT]`, or not
3. Within each group, count how many of those positions are mutated
4. Compare the two mutation rates. Is the hotspot rate higher, and by how much?

The alignments are lowercase, so uppercase each 3-mer before matching it.

In [ ]:
# your code here...

## Wrap-up

In this tutorial, you:

- Loaded and inspected IMGT/V-QUEST AIRR-formatted data
- Counted mutations per sequence by comparing an observed sequence to its inferred germline
- Used IMGT region boundaries to pull each region out of a sequence and compute its mutation density
- Found that the CDRs are mutated several times more heavily than the framework regions
- (Optionally) looked for AID hotspot motifs with regular expressions

Somatic hypermutation shapes B cell receptor sequences during affinity maturation, and it does so unevenly: concentrated in the loops that touch antigen, sparse in the scaffold that holds them. Separating that signal from the recombination that came before it is most of the work in reading this kind of data.